In [0]:
from pyspark.sql.functions import col, date_trunc, to_date, avg, min, max, count, round

df = spark.table("workspace.oura.silver_heartrate")

df_gold = df.groupBy(
    to_date(col("timestamp")).alias("day"),
    col("source")
).agg(
    round(avg(col("bpm")), 1).alias("avg_bpm"),
    min(col("bpm")).alias("min_bpm"),
    max(col("bpm")).alias("max_bpm"),
    count(col("bpm")).alias("sample_count")
)

df_gold.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.gold_heartrate_daily")
print(f"✓ gold_heartrate_daily: {df_gold.count()} rows")

✓ gold_heartrate_daily: 1923 rows


In [0]:
from pyspark.sql.functions import sum, countDistinct, when, lit

df = spark.table("workspace.oura.silver_workout")

df_gold = df.groupBy("day").agg(
    round(sum(col("calories")), 1).alias("total_calories"),
    round(sum(col("duration_minutes")), 1).alias("total_duration_minutes"),
    count(col("id")).alias("workout_count")
).withColumn("had_workout", lit(True))

df_gold.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.gold_workout_summary")
print(f"✓ gold_workout_summary: {df_gold.count()} rows")

✓ gold_workout_summary: 64 rows


In [0]:
cardio   = spark.table("workspace.oura.silver_cardiovascular_age")
ready    = spark.table("workspace.oura.silver_readiness")
stress   = spark.table("workspace.oura.silver_stress")
hr_daily = spark.table("workspace.oura.gold_heartrate_daily")
workout  = spark.table("workspace.oura.gold_workout_summary")

# Pivot heartrate to get awake and sleep as separate columns
from pyspark.sql.functions import when

hr_awake = hr_daily.filter(col("source") == "awake").select(
    col("day"),
    col("avg_bpm").alias("avg_bpm_awake"),
    col("min_bpm").alias("min_bpm_awake"),
    col("max_bpm").alias("max_bpm_awake")
)

hr_sleep = hr_daily.filter(col("source") == "sleep").select(
    col("day"),
    col("avg_bpm").alias("avg_bpm_sleep"),
    col("min_bpm").alias("min_bpm_sleep")
)

df_gold = cardio \
    .join(ready,    "day", "left") \
    .join(stress,   "day", "left") \
    .join(hr_awake, "day", "left") \
    .join(hr_sleep, "day", "left") \
    .join(workout,  "day", "left") \
    .select(
        cardio.day,
        cardio.vascular_age,
        cardio.pulse_wave_velocity,
        ready.readiness_score,
        ready.temperature_deviation,
        ready.activity_balance,
        ready.body_temperature,
        ready.hrv_balance,
        ready.previous_day_activity,
        ready.previous_night,
        ready.recovery_index,
        ready.resting_heart_rate,
        ready.sleep_balance,
        stress.stress_high,
        stress.recovery_high,
        hr_awake.avg_bpm_awake,
        hr_awake.min_bpm_awake,
        hr_awake.max_bpm_awake,
        hr_sleep.avg_bpm_sleep,
        hr_sleep.min_bpm_sleep,
        workout.total_calories,
        workout.total_duration_minutes,
        workout.workout_count,
        workout.had_workout
    )

df_gold.write.format("delta").mode("overwrite").saveAsTable("workspace.oura.gold_daily_summary")
print(f"✓ gold_daily_summary: {df_gold.count()} rows")

✓ gold_daily_summary: 687 rows


In [0]:
%sql
SELECT
  DATE_TRUNC('quarter', day) AS quarter,
  ROUND(AVG(vascular_age), 1) AS avg_vascular_age,
  ROUND(AVG(stress_high), 0) AS avg_stress,
  ROUND(AVG(min_bpm_awake), 1) AS avg_min_bpm,
  COUNT(*) AS days
FROM workspace.oura.gold_daily_summary
GROUP BY 1
ORDER BY 1

quarter,avg_vascular_age,avg_stress,avg_min_bpm,days
2024-04-01T00:00:00.000Z,23.4,2420.0,59.3,45
2024-07-01T00:00:00.000Z,24.0,1937.0,59.0,92
2024-10-01T00:00:00.000Z,24.5,1385.0,60.4,91
2025-01-01T00:00:00.000Z,25.5,1960.0,61.5,90
2025-04-01T00:00:00.000Z,25.5,1982.0,62.3,89
2025-07-01T00:00:00.000Z,25.8,1996.0,61.7,92
2025-10-01T00:00:00.000Z,25.9,2421.0,61.5,84
2026-01-01T00:00:00.000Z,26.9,1463.0,61.4,72
2026-04-01T00:00:00.000Z,27.6,4725.0,60.8,32
